# Practice — Trace a Python Request
AA-01-02-E05 · AI with review

Run in order. Predict before running each request. Supplied mode needs no key. OpenAI mode uses the same .env setup and request fields as Section 2.3. The request log contains instructions, input and model, never your key.


## OpenAI setup

Create the configuration used in Section 2.3. The cell below creates a blank `openai.env.example` file without replacing an existing file. Open it in the notebook file browser, save a copy as `.env` at the package root, then manually replace both placeholders with your own key and model name. Do not put the key in a code cell or share the filled file.

A standard Python kernel needs `openai` and `python-dotenv` (`%pip install openai python-dotenv`). Set `MODE = "live"` to run your prompts through OpenAI. The Try It browser kernel uses supplied responses in `"offline"` mode; it cannot run the standard OpenAI SDK. A supplied response does not test an edited prompt.


## Start here

This notebook includes its own setup. You do not need to run another notebook first. Start at the top and run each code cell in order, including the setup cells. To run everything, choose **Run → Run All Cells**. After restarting the kernel, run the setup again before running the experiment.

The default mode uses supplied examples and needs no API key. For OpenAI requests, follow the configuration instructions in this notebook and use a standard Python kernel.


In [ ]:
from pathlib import Path

config_template = Path("openai.env.example")
if not config_template.exists():
    config_template.write_text("OPENAI_API_KEY=YOUR_OPENAI_API_KEY\nOPENAI_MODEL=YOUR_MODEL_NAME\n", encoding="utf-8")
# The downloaded package uses one configuration at its root.
package_root = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "COMPANION_VERSION").is_file()), None)
if package_root:
    print("Configure the package-root .env beside COMPANION_VERSION.")
else:
    print("Open openai.env.example, save a copy as .env, and replace the placeholders there.")


In [ ]:
from types import SimpleNamespace

# Supplied examples only. No network calls or API credentials.
SAVED_RESPONSES = {
    "What is a variable in Python?": "A variable is a name that refers to a value. For example, age = 25.",
    "Explain what a Python list is to a beginner. Provide a simple Python example.": "A list holds a sequence of items. For example, books = ['History', 'Science']."
}
request_log = []

class SampleResponses:
    def create(self, *, model, instructions, input):
        text = SAVED_RESPONSES.get(" ".join(input.split()))
        if text is None:
            text = "No saved example matches this request. No AI model was called; save it for an instructor demonstration."
        return SimpleNamespace(output_text="[Supplied example — no API call]\n" + text)

MODE = "offline"  # Set to "live" after preparing .env in a standard Python kernel.
MODEL = "YOUR_MODEL_NAME"
if MODE == "live":
    import sys
    if sys.platform in {"emscripten", "wasi"}:
        raise RuntimeError("The OpenAI SDK needs a standard Python kernel. Use offline mode in browser Python.")
    import os
    from openai import OpenAI
    from dotenv import load_dotenv
    load_dotenv()
    MODEL = os.getenv("OPENAI_MODEL", "")
    if not MODEL or MODEL.startswith("YOUR_"):
        raise ValueError("Set OPENAI_MODEL in your .env file.")
    client = OpenAI(timeout=45.0, max_retries=0)
else:
    client = SimpleNamespace(responses=SampleResponses())


Predict which values become instructions and input in the request. Define the reusable function, then run the example.

In [ ]:
def get_completion(system_prompt, user_prompt):
    request_log.append({"model": MODEL, "instructions": system_prompt, "input": user_prompt, "mode": MODE})
    response = client.responses.create(
        model=MODEL,
        instructions=system_prompt,
        input=user_prompt
    )
    return response.output_text


In [ ]:
system_prompt = """You are a friendly Python tutor.
Use simple language and one practical example."""
user_prompt = "What is a variable in Python?"
response = get_completion(system_prompt, user_prompt)
print(response)


Change only the user request. Predict which field will change, then run the next cell.

In [ ]:
user_prompt = """Explain what a Python list is to a beginner.
Provide a simple Python example."""
response = get_completion(system_prompt, user_prompt)
print(response)


Inspect the latest request. Explain why the system instructions stay the same.

In [ ]:
print(request_log[-1])

Run these checks. They verify request wiring, not AI response quality.

In [ ]:
assert request_log[-1]["instructions"] == system_prompt
assert request_log[-1]["input"] == user_prompt
assert request_log[-1]["model"] == MODEL
print("The latest request contains the intended instructions, user input and model field.")


Change the request to a new question and run again. Explain why the offline substitute cannot produce a new model response. Submit the two request dictionaries and a two-sentence explanation. For a fresh response, configure .env and use live mode in a standard Python kernel.

### A first OpenAI request

The following optional cell follows the book's request pattern. After preparing `.env` in a standard Python kernel, set `RUN_OPENAI_EXAMPLE = True` and run it. It makes one actual API request. Leave it False for supplied browser practice. Later exercises use `get_completion()` with separate system instructions and user requests.


In [ ]:
RUN_OPENAI_EXAMPLE = False
if RUN_OPENAI_EXAMPLE:
    import sys
    if sys.platform in {"emscripten", "wasi"}:
        raise RuntimeError("Run this OpenAI cell in a standard Python kernel; browser practice uses supplied responses.")
    import os
    from openai import OpenAI
    from dotenv import load_dotenv

    load_dotenv()
    client = OpenAI()
    response = client.responses.create(
        model=os.getenv("OPENAI_MODEL"),
        input="Explain what an AI agent is in a few sentences."
    )
    print(response.output_text)
    client.close()
